# Driving policy first: Expected SARSA, DQN and Dyna-DQN

Learn **five driving actions** while all 18 sensors stay on. This is a separate,
short overtaking experiment; previous studies, stopped workers, SNNs, and saved
models remain unchanged. No sensor scheduling is learned in this notebook.

The initial task is **1,000 m within 45 s**, with one slower lead vehicle and an
initially clear passing lane. Success requires progress past the lead vehicle;
remaining behind its 18–20 m/s motion cannot finish on time. Agents receive real
sensor observations with noise, latency, warm-up and aging, not traffic truth.

The notebook executes a **12-episode-per-agent pilot** and five frozen test trips
per agent. This is an integration/stability check, not evidence of convergence.
The optional larger experiment is **500 episodes × 3 agents × 1 seed = 1,500
training episodes**, and **does not run unless `RUN_TRAINING=True`**. No background
process is launched. Existing STOP markers are respected by leaving those studies alone.

Select **MORL4VSO (.venv)**. Dependencies are already supplied by
`../requirements-dqn.txt`. Outputs are written to fresh timestamped folders under
`experiments/artifacts/driving_policy_v1/`.

In [1]:
from pathlib import Path
import os, sys, json
from dataclasses import asdict, replace
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "environments/sensor_highway/driving.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from environments.sensor_highway.driving import driving_config, DrivingEnv
from environments.sensor_highway.overtaking import oracle_action
from algorithms.driving_agents import DrivingSettings, DrivingEncoder
from experiments.driving_policy_helpers import new_output, compare_driving, rollout_driving, AGENTS, LABELS

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
config = driving_config()
settings = DrivingSettings(episodes=500)
TRAIN_SEED = 42
TEST_SEEDS = list(range(100000, 100020))
PILOT_EPISODES = 12
PILOT_TEST_SEEDS = TEST_SEEDS[:5]
RUN_PILOT = True
RUN_TRAINING = False
assert not set(range(TRAIN_SEED, TRAIN_SEED + settings.episodes)).intersection(TEST_SEEDS)
print("Interpreter:", sys.executable)
print("Full optional budget:", settings.episodes * len(AGENTS), "training episodes; one training seed")
display(pd.Series(asdict(settings), name="Driving learning settings"))
display(pd.Series(asdict(config.reward), name="Shared reward coefficients"))

Interpreter: /Users/zac/Desktop/Reinforcment_Learning/final_project/.venv/bin/python
Full optional budget: 1500 training episodes; one training seed


episodes                       500
gamma                         0.99
alpha                         0.12
epsilon_start                  1.0
epsilon_end                    0.1
learning_rate               0.0001
hidden_size                     64
batch_size                      32
replay_capacity              10000
learning_starts                128
target_update_steps            100
planning_steps                   1
model_learning_starts         1000
double_q                      True
validation_interval             50
min_validation_terminals         5
feature_mae_limit             0.05
reward_mae_limit               0.1
terminal_mae_limit            0.15
q_abort_limit                 20.0
Name: Driving learning settings, dtype: object

route_distance_m          1000.000
progress_weight              0.001
time_weight                  0.002
completion_bonus             1.000
early_completion_bonus       0.100
failure_penalty              2.000
energy_weight                0.000
uncertainty_weight           0.000
Name: Shared reward coefficients, dtype: float64

## What changes from the failed long-route experiment

- **Fixed sensing:** driving is the only learned decision. Sensor energy is still
  measured, but both energy and uncertainty reward coefficients are zero.
- **Short horizon:** the task is 1,000 m / 45 s, with gamma 0.99 rather than 0.999.
- **Small rewards:** progress is 0.001/m, time cost 0.002/s, completion +1,
  early-completion bonus up to 0.1, and crash/deadline failure −2. The total
  positive reward budget is at most **2.1**, before costs and discounting.
- **Neural stability:** Adam learning rate is 0.0001. Both neural agents use a
  Double-DQN target by default (online action selection, frozen-network evaluation).
  Set `double_q=False` for a vanilla-DQN ablation in a fresh output folder.
- **Shared information:** the original sensor features are augmented with ego
  road localization, commanded target lane/speed, lane availability, lane-change
  activity, and the last driving command. Road localization is an explicit assumed
  measurement. No other vehicle's true position, speed, gap, or identity is included.
- **Diagnostic guard:** abort if observed Q-values are nonfinite or exceed 20 in
  absolute magnitude. The guard reports a failure; it does not clip learned Q-values.

This package changes several aspects to establish a learnable baseline. It is not
an ablation attributing improvement to a single change. Later comparisons should
vary one setting at a time. Linear Expected SARSA remains linear; it is not a
neural network. Its original normalized semi-gradient update is retained.

## Always-on sensing and warm-up

Every episode begins with one forced **0.5 s IDLE** interval that requests every
sensor on. Timing, distance, energy and reward from this interval count toward
the episode, but the forced action is not a learned transition. The first learned
decision therefore receives actual sensor measurements instead of an empty map.
All subsequent actions keep sensors on. Real sensor limitations remain present.

The ego starts at 25 m/s. A same-lane lead car starts 80–90 m ahead at a seeded
speed of 18–20 m/s and holds its lane. Ego target speeds are 15, 20, 25 and 30 m/s;
FASTER/SLOWER change target speed, while LANE_LEFT/RIGHT and IDLE retain their
usual meanings. One overtake is counted after clearing the lead car by 10 m.

The new observation schema prevents accidentally loading models trained with the
older observation representation. Model outputs retain driving/sensing action
labels for future integration, but every action's sensing label is `all_on`.

In [2]:
with DrivingEnv(config) as env:
    obs, info = env.reset(seed=TRAIN_SEED)
    encoder = DrivingEncoder(config)
    features = encoder(obs)
    assert env.action_space.n == 5
    assert env.observation_space.contains(obs)
    assert np.all(obs["sensors"][:, 0] == 1)
    assert np.all(obs["sensors"][:, 1] == 1)
    assert info["time_s"] == 0.5
    print("Features:", len(features), "| driving actions:", env.action_space.n)
    print("First decision at", info["time_s"], "s; requested sensors:", int(obs["sensors"][:,0].sum()))
    print("Observation keys:", list(obs))

Features: 123 | driving actions: 5
First decision at 0.5 s; requested sensors: 18
Observation keys: ['grid', 'valid_cells', 'sensors', 'ego', 'task', 'controller']


## Scenario feasibility checks

Development-only scripted controllers establish that overtaking is feasible and
following the lead car or repeatedly slowing cannot satisfy the deadline. The
scripted controller reads simulator truth and is **only a diagnostic oracle**.
It supplies neither demonstrations nor observations to the learned agents.

In [3]:
baseline_rows = []
for development_seed in [900001, 900002, 900003]:
    for controller in ["oracle", "follow", "slow"]:
        with DrivingEnv(config) as env:
            obs, info = env.reset(seed=development_seed)
            while True:
                if controller == "slow":
                    action = 4
                else:
                    action = oracle_action(env, allow_passing=controller == "oracle")["drive"]
                obs, reward, terminated, truncated, info = env.step(action)
                if terminated or truncated:
                    break
            baseline_rows.append({"controller": controller, "seed": development_seed,
                "completed": info["route_completed"], "crashed": info["crashed"],
                "elapsed_s": info["time_s"], "distance_m": info["distance_m"],
                "overtakes": info["overtakes"]})
validation = pd.DataFrame(baseline_rows)
assert validation[validation.controller == "oracle"].completed.all()
assert not validation[validation.controller == "oracle"].crashed.any()
assert (validation[validation.controller == "oracle"].overtakes == 1).all()
assert not validation[validation.controller != "oracle"].completed.any()
display(validation.round(3))

,controller,seed,completed,crashed,elapsed_s,distance_m,overtakes
0,oracle,900001,True,False,34.0,1011.750,1
1,follow,900001,False,False,45.0,919.308,0
2,slow,900001,False,False,45.0,688.500,0
3,oracle,900002,True,False,34.0,1011.750,1
4,follow,900002,False,False,45.0,881.449,0
5,slow,900002,False,False,45.0,688.500,0
6,oracle,900003,True,False,34.0,1011.750,1
7,follow,900003,False,False,45.0,871.448,0
8,slow,900003,False,False,45.0,688.500,0


## Dyna-DQN planning safeguards

The Q network learns from every real transition. Every tenth real transition is
reserved from **model training** for model validation. The other 90% train the
feature-change, reward and terminal predictor. Model sampling has a separate random
stream, so with planning disabled the real Q updates match DQN on identical data.

Planning requires at least 1,000 model-training transitions, at least five terminal
and 20 nonterminal validation transitions, and held-out error thresholds for
features, rewards and both terminal classes. A low error caused only by predicting
"not terminal" cannot pass the gate. At most one imagined update accompanies each
real update after the gate passes. Model-generated rewards are bounded to the task's
reward range. Imagined experience never enters real replay.

During this tiny pilot, planning may remain disabled. Report that explicitly;
training a predictor alone is not evidence that planning improves the policy.
Set `planning_steps=0` for a matched control. Portable saved Q policies are for
inference; each trusted local `resume.pkl` also preserves training/model state
for future work. Automatic training resume is not implemented in this notebook.

## Untrained policy control

Evaluate the same initialized policies before training on the same five pilot test
scenarios. This distinguishes success after training from behavior already present
in random network weights. Do not use these five test scenarios to tune parameters;
use new held-out scenarios for subsequent development decisions.

In [4]:
untrained_evaluation = pd.DataFrame()
if RUN_PILOT:
    rows = []
    for policy, agent_class in AGENTS.items():
        initial_agent = (agent_class(len(features)) if policy == "sarsa" else
                         agent_class(len(features), replace(settings, episodes=PILOT_EPISODES), TRAIN_SEED))
        rows += [{"policy": LABELS[policy], **rollout_driving(config, initial_agent, seed)[0]}
                 for seed in PILOT_TEST_SEEDS]
    untrained_evaluation = pd.DataFrame(rows)
    display(untrained_evaluation.groupby("policy", sort=False).agg(
        trips=("completed", "size"), completion=("completed", "mean"), crashes=("crashed", "sum")))

,trips,completion,crashes
policy,,,
Expected SARSA (linear),5,0.2,3
DQN (Double target),5,0.0,0
Dyna-DQN (gated planning),5,0.0,0


In [5]:
pilot_output = None
if RUN_PILOT:
    pilot_settings = replace(settings, episodes=PILOT_EPISODES)
    pilot_output = new_output(config, pilot_settings, "pilot", TRAIN_SEED)
    validation.to_csv(pilot_output / "scenario_validation.csv", index=False)
    untrained_evaluation.to_csv(pilot_output / "untrained_evaluation.csv", index=False)
    pilot_agents, pilot_training, pilot_evaluation, pilot_summary = compare_driving(
        config, pilot_settings, pilot_output, seed=TRAIN_SEED,
        test_seeds=PILOT_TEST_SEEDS, replays=True)
    print("Pilot saved:", pilot_output)
    display(pilot_summary.round(4))
    before_after = untrained_evaluation.groupby("policy").completed.mean().rename("untrained_completion").to_frame().join(
        pilot_summary.completion.rename("trained_completion"))
    before_after.to_csv(pilot_output / "before_after.csv")
    display(before_after)
    display(pilot_training.groupby("policy", sort=False).agg(
        episodes=("episode", "size"), training_completions=("completed", "sum"),
        max_abs_q=("max_abs_q", "max"), final_planning_updates=("planning_updates", "max")))
else:
    print("Pilot disabled.")

Expected SARSA (linear): 12/12, return=-1.400, max|Q|=0.222, wall=9.3s


DQN (Double target): 12/12, return=1.956, max|Q|=0.317, wall=7.5s


Dyna-DQN (gated planning): 12/12, return=1.956, max|Q|=0.317, wall=7.9s


Pilot saved: /Users/zac/Desktop/Reinforcment_Learning/final_project/experiments/artifacts/driving_policy_v1/pilot_20261121_102702_395957


,trips,completion,crash,deadline_failure,successful_time_s,mean_distance_m,mean_overtakes,max_abs_q
policy,,,,,,,,
Expected SARSA (linear),5,0.0,1.0,0.0,NaN,339.0953,0.0,0.2204
DQN (Double target),5,1.0,0.0,0.0,34.4,1011.0499,1.0,0.3383
Dyna-DQN (gated planning),5,1.0,0.0,0.0,34.4,1011.0499,1.0,0.3383


,untrained_completion,trained_completion
policy,,
DQN (Double target),0.0,1.0
Dyna-DQN (gated planning),0.0,1.0
Expected SARSA (linear),0.2,0.0


,episodes,training_completions,max_abs_q,final_planning_updates
policy,,,,
Expected SARSA (linear),12,4,0.290919,0
DQN (Double target),12,4,0.317197,0
Dyna-DQN (gated planning),12,4,0.317197,0


In [6]:
if pilot_output is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for label, group in pilot_training.groupby("policy", sort=False):
        axes[0].plot(group.episode, group["return"], marker=".", label=label)
        axes[1].plot(group.episode, group.max_abs_q, marker=".", label=label)
    axes[0].set(title="Pilot training return", xlabel="Episode", ylabel="Return")
    axes[1].set(title="Pilot value stability", xlabel="Episode", ylabel="Maximum observed |Q|")
    axes[0].legend(fontsize=8)
    fig.suptitle("Integration pilot only — not converged driving policies")
    fig.savefig(pilot_output / "pilot_diagnostics.png", dpi=160)
    display(fig)
    plt.close(fig)
    for policy, agent in pilot_agents.items():
        if hasattr(agent, "model_diagnostics"):
            display(pd.Series({"policy": LABELS[policy], "planning_updates": agent.planning_updates,
                               "planning_allowed": agent.planning_allowed, **agent.model_diagnostics}))
    for replay in sorted(pilot_output.glob("*/replay/replay.html")):
        relative = os.path.relpath(replay, ROOT / "experiments")
        display(Markdown(f"[Pilot replay: {replay.parent.parent.name}]({relative})"))

<Figure size 1200x400 with 2 Axes>

policy                  Dyna-DQN (gated planning)
planning_updates                                0
planning_allowed                            False
model_feature_mae                        0.062355
model_reward_mae                         0.067956
model_terminal_mae                       0.950963
validation_terminals                            2
dtype: object

[Pilot replay: dqn](artifacts/driving_policy_v1/pilot_20261121_102702_395957/dqn/replay/replay.html)

[Pilot replay: dyna_dqn](artifacts/driving_policy_v1/pilot_20261121_102702_395957/dyna_dqn/replay/replay.html)

[Pilot replay: sarsa](artifacts/driving_policy_v1/pilot_20261121_102702_395957/sarsa/replay/replay.html)

## Optional 500-episode driving experiment

**Disabled by default.** Setting `RUN_TRAINING=True` runs 500 fresh training episodes
for each of three agents on one seed (1,500 episodes total), then 20 frozen test
trips per agent in clear weather. This stage tests a simple driving task; weather
variation, neighboring traffic, multiple passes and longer routes are later stages.

Every run receives a new output folder. Training checkpoints are written every
25 episodes and at the end. Closing a running training kernel stops this foreground
experiment; no worker is launched in the background. Previous stopped 9 km runs
remain stopped. Changing this flag does not resume or overwrite those studies.

Do not promote a policy to harder tasks based on training return alone. Inspect
held-out completion/crashes, overtakes, deadline failures, Q stability, prediction
errors and the replay. One seed is preliminary, not a robustness result.

In [7]:
training_output = None
if RUN_TRAINING:
    training_output = new_output(config, settings, "driving_500ep", TRAIN_SEED)
    validation.to_csv(training_output / "scenario_validation.csv", index=False)
    agents, training, evaluation, summary = compare_driving(
        config, settings, training_output, seed=TRAIN_SEED,
        test_seeds=TEST_SEEDS, replays=True)
    print("Training results:", training_output)
    display(summary.round(4))
else:
    print("500-episode training has not been launched. Set RUN_TRAINING=True to run it.")

500-episode training has not been launched. Set RUN_TRAINING=True to run it.


## Later: integrate sensor optimization

Once driving is reliable, expand the five driving choices to the 20 joint driving/
sensing choices and continue learning. Keep driving trainable as observation quality
changes. Introduce energy costs in a separate controlled experiment and compare
with joint training from scratch. Dyna's action-conditioned model must be adapted
and retrained for sensor transitions before planning with those actions. This
notebook does not implement that second stage or imply that joint learning is impossible.